# 20 Newsgroups → LTS pipeline (cleaned Colab notebook)

This notebook is organized as a single pipeline:

1. Mount Drive and set paths  
2. Parse the 20 Newsgroups text files  
3. Build the binary target dataset (for example `sci.med` vs rest)  
4. Create train / validation / test CSVs in LTS format  
5. Create few-shot examples for the labeler  
6. Write the cleaned Python modules needed by the pipeline  
7. Reset sampler state files before a fresh run  
8. Run `main_cluster.py`

The code below keeps only the changes that are needed for 20 Newsgroups and removes fragile edits that can break the pipeline.

This version also preserves the original dataset ground truth as `true_label` in the sampled labeled CSV, while keeping Qwen's pseudo-label in `answer`/`label` for auditing.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project"
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)
print("Working directory:", os.getcwd())
print("Files:", sorted(os.listdir(BASE_PATH))[:20])

Working directory: /content/drive/MyDrive/Colab Notebooks/Data_Engineering_project
Files: ['LDA.py', '__pycache__', 'alt.atheism.txt', 'cleaned_20newsgroups_lts_pipeline_llm_text_shortening_fixed_csv.ipynb', 'cleaned_20newsgroups_lts_pipeline_qwen_stricter_prompt.ipynb', 'cleaned_20newsgroups_lts_pipeline_qwen_true_label_audit.ipynb', 'cleaned_20newsgroups_lts_pipeline_rec_autos_updated.ipynb', 'cleaned_20newsgroups_lts_pipeline_title_subject_only.ipynb', 'cleaned_20newsgroups_lts_pipeline_title_subject_only_step5_fixed.ipynb', 'comp.graphics.txt', 'comp.os.ms-windows.misc.txt', 'comp.sys.ibm.pc.hardware.txt', 'comp.sys.mac.hardware.txt', 'comp.windows.x.txt', 'data', 'few_shot_examples_sci_med.json', 'fine_tune.py', 'labeling.py', 'list.csv', 'log']


In [3]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")

Enter your HF token: ··········
HF token set.


## 1) Install dependencies

This version uses **scikit-learn LDA** for clustering instead of a gensim-based implementation.  
That keeps the notebook lighter and avoids extra dependency issues on Colab.

In [4]:
!pip -q install -U transformers datasets accelerate sentencepiece scikit-learn nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 178.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 35.1 MB/s eta 0:00:00


In [5]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2) Parse the 20 Newsgroups files

This expects the 20 source files to already be in `BASE_PATH`, one file per class, such as:

- `alt.atheism.txt`
- `sci.med.txt`
- ...

Each file is parsed into records with:
`newsgroup`, `document_id`, `from`, `subject`, `body`, `text`

In [6]:
import os
import re
import pandas as pd
import unicodedata

HEADER_KEYS = {"newsgroup", "document_id", "from", "subject"}

def _clean_field(text):
    text = str(text or "")
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\x00", " ")
    text = "".join(ch for ch in text if ch == "\n" or ch == "\t" or ch.isprintable())
    return text.strip()

def parse_newsgroup_file(filepath):
    filename = os.path.basename(filepath)
    fallback_newsgroup = filename.replace(".txt", "")

    with open(filepath, "r", encoding="latin-1", errors="ignore") as f:
        content = f.read()

    chunks = re.split(r'(?mi)(?=^Newsgroup:\s)', content)
    records = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        lines = chunk.splitlines()
        record = {
            "newsgroup": fallback_newsgroup,
            "document_id": "",
            "from": "",
            "subject": "",
            "body": "",
            "text": ""
        }

        body_start = None
        last_header_idx = -1

        for i, raw_line in enumerate(lines):
            line = raw_line.rstrip("\n")
            stripped = line.strip()

            if body_start is None and stripped == "":
                body_start = i + 1
                break

            m = re.match(r'(?i)^([A-Za-z_]+):\s*(.*)$', stripped)
            if m:
                key = m.group(1).lower()
                value = m.group(2).strip()
                if key in HEADER_KEYS:
                    record[key] = value
                    last_header_idx = i
                    continue

            # If we hit a non-header line before a blank line, treat the remainder as body
            if last_header_idx >= 0:
                body_start = i
                break

        if body_start is None:
            body_start = last_header_idx + 1 if last_header_idx >= 0 else 0

        body_lines = lines[body_start:]
        body = "\n".join(body_lines).strip()
        record["body"] = body

        if record["subject"] and record["body"]:
            record["text"] = record["subject"] + "\n\n" + record["body"]
        elif record["subject"]:
            record["text"] = record["subject"]
        else:
            record["text"] = record["body"]

        for col in ["newsgroup", "document_id", "from", "subject", "body", "text"]:
            record[col] = _clean_field(record[col])

        if record["document_id"] != "":
            records.append(record)

    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=["newsgroup","document_id","from","subject","body","text"])

    df = df.drop_duplicates(subset=["document_id"]).reset_index(drop=True)
    return df

In [7]:
txt_files = sorted([
    f for f in os.listdir(BASE_PATH)
    if f.endswith(".txt")
])

print("Found", len(txt_files), "txt files")
print(txt_files[:10])

all_dfs = []
for filename in txt_files:
    filepath = os.path.join(BASE_PATH, filename)
    file_df = parse_newsgroup_file(filepath)
    all_dfs.append(file_df)
    print(f"{filename}: {len(file_df)} documents")

raw_df = pd.concat(all_dfs, ignore_index=True)
print("\nTotal parsed rows:", len(raw_df))
print("Unique document_id:", raw_df["document_id"].nunique())
print(raw_df.head(2))

Found 20 txt files
['alt.atheism.txt', 'comp.graphics.txt', 'comp.os.ms-windows.misc.txt', 'comp.sys.ibm.pc.hardware.txt', 'comp.sys.mac.hardware.txt', 'comp.windows.x.txt', 'misc.forsale.txt', 'rec.autos.txt', 'rec.motorcycles.txt', 'rec.sport.baseball.txt']
alt.atheism.txt: 799 documents
comp.graphics.txt: 973 documents
comp.os.ms-windows.misc.txt: 985 documents
comp.sys.ibm.pc.hardware.txt: 982 documents
comp.sys.mac.hardware.txt: 961 documents
comp.windows.x.txt: 980 documents
misc.forsale.txt: 972 documents
rec.autos.txt: 990 documents
rec.motorcycles.txt: 994 documents
rec.sport.baseball.txt: 994 documents
rec.sport.hockey.txt: 999 documents
sci.crypt.txt: 991 documents
sci.electronics.txt: 981 documents
sci.med.txt: 990 documents
sci.space.txt: 987 documents
soc.religion.christian.txt: 997 documents
talk.politics.guns.txt: 910 documents
talk.politics.mideast.txt: 940 documents
talk.politics.misc.txt: 775 documents
talk.religion.misc.txt: 628 documents

Total parsed rows: 18828
U

## 3) Build the binary classification dataset and a shortened `title` field

In [8]:
TARGET_CLASS = "rec.autos"
TARGET_SLUG = TARGET_CLASS.replace(".", "_")

def looks_like_encoded_line(line):
    stripped = line.strip()
    if len(stripped) > 80 and re.fullmatch(r"[A-Za-z0-9+/=_-]+", stripped):
        return True
    if re.match(r"^(begin\s+\d{3}\s+|M[A-Z0-9`!\"#$%&'()*+,\-./:;<=>?@\[\]^_`{|}~]{20,})", stripped):
        return True
    return False

def build_llm_text(subject, body, full_text="", max_chars=1000):
    subject = _clean_field(subject)
    body = _clean_field(body)
    full_text = _clean_field(full_text)

    source_text = body if body else full_text

    cleaned_lines = []
    for line in source_text.splitlines():
        stripped = line.strip()

        if not stripped:
            continue
        if stripped.startswith(">"):
            continue
        if re.match(r"^(In article .*writes:|.*writes:)$", stripped):
            continue
        if re.match(
            r"^(From:|Subject:|Organization:|Lines:|NNTP-Posting-Host:|Reply-To:|Distribution:|Keywords:|Summary:|Archive-name:|Last-modified:|Newsgroup:|document_id:)",
            stripped,
            flags=re.IGNORECASE
        ):
            continue
        if re.fullmatch(r"[-_=~*`]{4,}", stripped):
            continue
        if looks_like_encoded_line(stripped):
            continue

        cleaned_lines.append(stripped)

    cleaned_body = " ".join(cleaned_lines)
    cleaned_body = re.sub(r"\s+", " ", cleaned_body).strip()

    if subject and cleaned_body:
        combined = f"{subject}\n\n{cleaned_body}"
    elif subject:
        combined = subject
    elif cleaned_body:
        combined = cleaned_body
    else:
        combined = full_text[:max_chars]

    return combined[:max_chars]

task_df = raw_df.copy()
task_df["binary_label"] = (task_df["newsgroup"] == TARGET_CLASS).astype(int)

task_df["raw_subject"] = task_df.get("subject", "").fillna("").astype(str)
task_df["raw_body"] = task_df.get("body", "").fillna("").astype(str)
task_df["raw_text"] = task_df.get("text", "").fillna("").astype(str)

task_df["llm_text"] = task_df.apply(
    lambda r: build_llm_text(
        r.get("subject", ""),
        r.get("body", ""),
        r.get("text", ""),
        max_chars=1000
    ),
    axis=1
)

# fallback if any rows still ended up empty
empty_mask = task_df["llm_text"].str.strip() == ""
task_df.loc[empty_mask, "llm_text"] = (
    task_df.loc[empty_mask, "raw_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 1000)
)

# make title the one text column used by the pipeline
task_df["title"] = task_df["llm_text"]

dedup_df = (
    task_df.groupby("document_id", as_index=False)
           .agg({
               "raw_text": "first",
               "raw_subject": "first",
               "raw_body": "first",
               "title": "first",
               "newsgroup": "first",
               "binary_label": "max"
           })
)

print("Deduplicated shape:", dedup_df.shape)
print(dedup_df["binary_label"].value_counts())
print(dedup_df["binary_label"].value_counts(normalize=True))
print(dedup_df[["newsgroup", "binary_label", "raw_subject", "title"]].head(3))

Deduplicated shape: (15404, 7)
binary_label
0    14414
1      990
Name: count, dtype: int64
binary_label
0    0.935731
1    0.064269
Name: proportion, dtype: float64
                 newsgroup  binary_label  \
0  comp.os.ms-windows.misc             0   
1  comp.os.ms-windows.misc             0   
2  comp.os.ms-windows.misc             0   

                                      raw_subject  \
0  Re: Is "Kermit" available for Windows 3.0/3.1?   
1                                    WIN STORM PC   
2                         Re: Win31 & doublespace   

                                               title  
0  Re: Is "Kermit" available for Windows 3.0/3.1?...  
1  WIN STORM PC\n\nAnyone have any info. on the v...  
2  Re: Win31 & doublespace\n\n.Chris I can't see ...  


In [9]:
train_df, test_df = train_test_split(
    dedup_df,
    test_size=0.20,
    stratify=dedup_df["binary_label"],
    random_state=SEED
)

train_inner_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["binary_label"],
    random_state=SEED
)

train_inner_df = train_inner_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train_inner:", train_inner_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train_inner: (9858, 7)
val: (2465, 7)
test: (3081, 7)


In [10]:
import csv

def to_lts_format(df):
    out = df.rename(columns={
        "document_id": "id",
        "binary_label": "label",
        "raw_subject": "subject",
    }).copy()

    out["title"] = out["title"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    out["subject"] = out["subject"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

    keep_cols = ["id", "title", "subject", "label", "newsgroup"]
    return out[keep_cols]

train_inner_lts = to_lts_format(train_inner_df)
val_lts = to_lts_format(val_df)
test_lts = to_lts_format(test_df)

train_inner_path = os.path.join(BASE_PATH, f"train_inner_20news_{TARGET_SLUG}.csv")
val_path = os.path.join(BASE_PATH, f"val_20news_{TARGET_SLUG}.csv")
test_path = os.path.join(BASE_PATH, f"test_20news_{TARGET_SLUG}.csv")

train_inner_lts.to_csv(train_inner_path, index=False, quoting=csv.QUOTE_ALL)
val_lts.to_csv(val_path, index=False, quoting=csv.QUOTE_ALL)
test_lts.to_csv(test_path, index=False, quoting=csv.QUOTE_ALL)

print(train_inner_path)
print(val_path)
print(test_path)
print(train_inner_lts["label"].value_counts())
print(train_inner_lts[["id", "newsgroup", "label", "subject", "title"]].head(2))


/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project/train_inner_20news_rec_autos.csv
/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project/val_20news_rec_autos.csv
/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project/test_20news_rec_autos.csv
label
0    9224
1     634
Name: count, dtype: int64
      id           newsgroup  label                             subject  \
0  54288    rec.sport.hockey      0     Re: ESPN Thumbs Up your $%#@*!!   
1  54658  talk.politics.guns      0  Re: Re: Guns GONE. Good Riddance !   

                                               title  
0  Re: ESPN Thumbs Up your $%#@*!! In article <19...  
1  Re: Re: Guns GONE. Good Riddance ! / iftccu:ta...  


## 3b) Optional smaller stratified debug split for Colab

This creates smaller train/validation files with the **same positive/negative proportion** as the full split.
Use these for quick Colab tests before running the full dataset on HPC.


In [11]:
DEBUG_FRACTION = 0.25  # 25% of each class; increase to 0.50 if you want a larger Colab test

def make_stratified_subset(df, frac, label_col="label", random_state=42):
    if frac >= 1.0:
        return df.copy().reset_index(drop=True)

    parts = []
    for label_value, group in df.groupby(label_col):
        n = max(1, int(round(len(group) * frac)))
        parts.append(group.sample(n=min(n, len(group)), random_state=random_state))
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=random_state).reset_index(drop=True)

train_inner_small = make_stratified_subset(train_inner_lts, DEBUG_FRACTION, random_state=SEED)
val_small = make_stratified_subset(val_lts, DEBUG_FRACTION, random_state=SEED)

train_inner_small_path = os.path.join(BASE_PATH, f"train_inner_20news_{TARGET_SLUG}_small.csv")
val_small_path = os.path.join(BASE_PATH, f"val_20news_{TARGET_SLUG}_small.csv")

train_inner_small.to_csv(train_inner_small_path, index=False, quoting=csv.QUOTE_ALL)
val_small.to_csv(val_small_path, index=False, quoting=csv.QUOTE_ALL)

print("Saved small debug files:")
print(train_inner_small_path, train_inner_small["label"].value_counts(normalize=True).to_dict(), len(train_inner_small))
print(val_small_path, val_small["label"].value_counts(normalize=True).to_dict(), len(val_small))


Saved small debug files:
/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project/train_inner_20news_rec_autos_small.csv {0: 0.9358766233766234, 1: 0.06412337662337662} 2464
/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project/val_20news_rec_autos_small.csv {0: 0.9351701782820098, 1: 0.06482982171799027} 617


In [12]:
train_ids = set(train_inner_lts["id"])
val_ids = set(val_lts["id"])
test_ids = set(test_lts["id"])

print("train/val overlap:", len(train_ids & val_ids))
print("train/test overlap:", len(train_ids & test_ids))
print("val/test overlap:", len(val_ids & test_ids))

train/val overlap: 0
train/test overlap: 0
val/test overlap: 0


## 4) Build a curated few-shot prompt set for the labeler (starter template for `rec.autos`)

This cell now writes a safe starter JSON for `rec.autos`. After you inspect the new train split, we can choose good positive and negative IDs together.

In [103]:
# Curated few-shot examples for rec.autos.
# This restores the exact working IDs/order from the earlier successful file.

CURATED_POSITIVE_IDS = [
    101558,  # 1993 Infiniti G20
    101637,  # Jeep Grand vs. Toyota 4-Runner
    103154,  # locking lugnuts / tire rebalance
    103176,  # Slick 50, any good?
    103212,  # Honda Mailing list?
    103812,  # Changing oil by yourself
    102837,  # Taurus/Sable rotor recall
    103377,  # legal car buying problems
]

CURATED_NEGATIVE_IDS = [
    104698,  # rec.motorcycles BMW mailing list
    104882,  # rec.motorcycles carb discussion
    104795,  # rec.motorcycles countersteering
    74810,   # misc.forsale car sale post
    39024,   # comp.graphics VESA driver
    38348,   # comp.graphics VGA/SVGA programming
]

def _clean_text(text, max_chars=140):
    return str(text).strip().replace("\n", " ")[:max_chars]

def build_curated_few_shot(df, positive_ids, negative_ids, id_col="id", text_col="title", label_col="label"):
    work = df.copy()
    work[id_col] = work[id_col].astype(str)
    chosen_ids = [str(x) for x in (positive_ids + negative_ids)]
    chosen = work[work[id_col].isin(chosen_ids)].copy()

    examples = []
    missing_ids = []

    for doc_id in chosen_ids:
        row = chosen[chosen[id_col] == doc_id]
        if row.empty:
            missing_ids.append(doc_id)
            continue

        row = row.iloc[0]
        doc_id_int = int(row[id_col])

        examples.append({
            "id": doc_id_int,
            "text": _clean_text(row[text_col], max_chars=140),
            "label": str(int(row[label_col]))
        })

    if missing_ids:
        print("Missing curated IDs from train split:", missing_ids)

    return examples

few_shot_examples = build_curated_few_shot(
    train_inner_lts,
    CURATED_POSITIVE_IDS,
    CURATED_NEGATIVE_IDS,
    id_col="id",
    text_col="title",
    label_col="label"
)

few_shot_path = os.path.join(BASE_PATH, f"few_shot_examples_{TARGET_SLUG}.json")

with open(few_shot_path, "w", encoding="utf-8") as f:
    json.dump(few_shot_examples, f, indent=2, ensure_ascii=False)

print("Saved few-shot examples to:", few_shot_path)
print("Number of curated examples:", len(few_shot_examples))
few_shot_examples[:5] if few_shot_examples else []

Saved few-shot examples to: /content/drive/MyDrive/Colab Notebooks/Data_Engineering_project/few_shot_examples_rec_autos.json
Number of curated examples: 14


[{'id': 101558,
  'text': '1993 Infiniti G20 I am thinking about getting an Infiniti G20. In consumer reports it is ranked high in many catagories including highest in',
  'label': '1'},
 {'id': 101637,
  'text': 'Re: Jeep Grand vs. Toyota 4-Runner hand. Land Crusier is just simply nice with shit-load of power and room. Fully stocked, it cost ~$40,000.',
  'label': '1'},
 {'id': 103154,
  'text': 'Re: locking lugnuts / tire rebalance?? Well, it depends on what kind of locking lugnuts you have. My previous car had locking lugnuts that w',
  'label': '1'},
 {'id': 103176,
  'text': "Re: Slick 50, any good? I don't have any written data but I know what I have experienced. I use S-50 in everything including my lawnmowers. ",
  'label': '1'},
 {'id': 103212,
  'text': 'Honda Mailing list? Is there a Honda mailing list, and if so how do I subscribe to it?',
  'label': '1'}]

## 5) Write the cleaned Python modules

This version is now pointed at `rec.autos`.

It also borrows two ideas worth keeping from your teammate's code:
- `fine_tune.py` prefers `training_text` when available and falls back to `title`
- `main_cluster.py` creates the labeler inside each iteration and frees GPU memory after labeling

In [15]:

preprocessing_code = r'''
import re
import pandas as pd

class TextPreprocessor:
    def clean_text(self, text):
        text = str(text)
        text = text.replace("\n", " ")
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def preprocess_df(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if "title" not in out.columns:
            raise ValueError("Expected a title column in the input dataframe.")
        out["title"] = out["title"].fillna("").astype(str)
        out["clean_title"] = out["title"].apply(self.clean_text)
        return out
'''
with open(os.path.join(BASE_PATH, "preprocessing.py"), "w", encoding="utf-8") as f:
    f.write(preprocessing_code)

print("Wrote preprocessing.py")


Wrote preprocessing.py


In [16]:
lda_code = r'''
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

class LDATopicModel:
    def __init__(self, num_topics=10, max_features=5000, random_state=42):
        self.num_topics = num_topics
        self.vectorizer = CountVectorizer(
            stop_words="english",
            max_df=0.95,
            min_df=2,
            max_features=max_features
        )
        self.model = LatentDirichletAllocation(
            n_components=num_topics,
            random_state=random_state
        )

    def fit_transform(self, documents):
        X = self.vectorizer.fit_transform(documents)
        topic_probs = self.model.fit_transform(X)
        return topic_probs.argmax(axis=1)
'''
with open(os.path.join(BASE_PATH, "LDA.py"), "w", encoding="utf-8") as f:
    f.write(lda_code)

print("Wrote LDA.py")

Wrote LDA.py


In [99]:
labeling_code = r"""
import pandas as pd
import torch
import os
import json
import re

from transformers import AutoModelForCausalLM, AutoTokenizer


class Labeling:
    def __init__(
        self,
        label_model="qwen",
        target_class="rec.autos",
        model_id="Qwen/Qwen2.5-3B-Instruct",
        few_shot_path=None
    ):
        self.label_model = label_model
        self.target_class = target_class
        self.model_id = model_id
        self.few_shot_path = few_shot_path
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.few_shot_examples = self.load_few_shot_examples()

    def load_few_shot_examples(self):
        if self.few_shot_path and os.path.exists(self.few_shot_path):
            with open(self.few_shot_path, "r", encoding="utf-8") as f:
                return json.load(f)
        return []

    def _clean_for_prompt(self, text, max_chars=200):
        text = str(text)
        text = re.sub(r'\S+@\S+', ' ', text)
        text = re.sub(r'\b\w+!\w+(?:!\w+)*\b', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars]

    def _build_examples_text(self):
        if not self.few_shot_examples:
            return ""

        selected = self.few_shot_examples[:8]

        lines = []
        for ex in selected:
            short_text = self._clean_for_prompt(ex["text"], max_chars=140)
            lines.append(f"Document: {short_text}")
            lines.append(f"Label: {ex['label']}")
            lines.append("")
        return "\n".join(lines).strip()

    def generate_prompt(self, title):
        if self.label_model == "qwen":
            return self.generate_prompt_qwen(title)
        elif self.label_model == "file":
            return None
        else:
            raise ValueError("Only 'qwen' or 'file' labeling is supported in this notebook.")

    def _base_prompt(self, title: str) -> str:
        examples = self._build_examples_text()
        short_title = self._clean_for_prompt(title, max_chars=200)

        prompt = f'''Task: classify a 20 Newsgroups post as rec.autos or not.

Label 1:
cars, trucks, SUVs, auto repair, driving, insurance for cars, buying cars, car ownership, maintenance, recalls, parts, tires, engines, transmissions, dealerships, used car advice, accidents involving cars.

Label 0:
motorcycles, pure for-sale ads, computers, graphics, software, medicine, religion, politics, science, or anything not mainly about cars.

Important:
- motorcycles = 0
- pure sale ads/classifieds = 0
- short replies still count as 1 if clearly about cars
- if unsure, output 0

Return only 1 or 0.
'''
        if examples:
            prompt += "\nExamples:\n" + examples + "\n\n"

        prompt += f"Document: {short_title}\nLabel:"
        return prompt

    def generate_prompt_qwen(self, title: str) -> str:
        return self._base_prompt(title.strip())

    def set_model(self):
        if self.label_model == "qwen":
            checkpoint = self.model_id
            self.model = AutoModelForCausalLM.from_pretrained(
                checkpoint,
                torch_dtype="auto",
                device_map="auto"
            )
            self.tokenizer = AutoTokenizer.from_pretrained(checkpoint)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print("Qwen model loaded")
        elif self.label_model == "file":
            self.model = None
            self.tokenizer = None
        else:
            raise ValueError("Only 'qwen' or 'file' labeling is supported in this notebook.")

    def predict_animal_product(self, row):
        label = Labeling.check_already_label(row)
        if label is not None:
            return label
        if self.label_model == "qwen":
            return self.get_qwen_label(row)
        elif self.label_model == "file":
            return self.get_file_label(row)
        else:
            raise ValueError("No supported label model selected")

    def generate_inference_data(self, data, column):
        if self.label_model != "file":
            examples = []
            for _, data_point in data.iterrows():
                raw_text = data_point[column]
                examples.append(
                    {
                        "id": data_point["id"],
                        "title": data_point["title"],
                        "training_text": data_point["clean_title"] if "clean_title" in data_point.index else data_point["title"],
                        "true_label": data_point["label"] if "label" in data_point.index else None,
                        "text": self.generate_prompt(raw_text),
                    }
                )
            data = pd.DataFrame(examples)
        return data

    def _extract_label(self, response_text: str) -> str:
        response_text = str(response_text).strip()

        match = re.search(r'\b([01])\b', response_text)
        if match:
            return match.group(1)

        lowered = response_text.lower()
        if lowered.startswith("1") or "label: 1" in lowered or "label 1" in lowered:
            return "1"
        if lowered.startswith("0") or "label: 0" in lowered or "label 0" in lowered:
            return "0"

        return "0"

    def get_qwen_label(self, row):
        user_prompt = row["text"]

        messages = [
            {
                "role": "system",
                "content": "You are a classifier. Reply with exactly one character: 1 or 0."
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer(
            [text],
            return_tensors="pt",
            truncation=True,
            max_length=768
        ).to(self.model.device)

        input_len = model_inputs.input_ids.shape[1]

        with torch.inference_mode():
            outputs = self.model.generate(
                **model_inputs,
                max_new_tokens=3,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = outputs[:, input_len:]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return self._extract_label(response)

    def get_file_label(self, row):
        raise NotImplementedError("File-based labeling is not implemented in this notebook.")

    @staticmethod
    def check_already_label(row):
        return None
"""
with open(os.path.join(BASE_PATH, "labeling.py"), "w", encoding="utf-8") as f:
    f.write(labeling_code)

print("Wrote labeling.py")

Wrote labeling.py


In [18]:
random_code = r'''
from typing import Any
import numpy as np
import pandas as pd

class RandomSampler:
    def __init__(self, n_bandits):
        self.n_bandits = n_bandits
        try:
            loaded = np.loadtxt("selected_ids.txt", dtype=str)
            self.selected_ids = set(np.atleast_1d(loaded).tolist())
        except Exception:
            self.selected_ids = set()

    def get_sample_data(self, df, sample_size, filter_label: bool, trainer: Any):
        def get_sample(data, size):
            if data.empty:
                return pd.DataFrame()
            return data.sample(min(size, len(data)), random_state=42)

        df = df.copy()
        df["id"] = df["id"].astype(str)
        df = df[~df["id"].isin(self.selected_ids)]

        if df.empty:
            raise ValueError("No unlabeled data left to sample from.")

        unique_clusters = sorted(df["label_cluster"].unique().tolist())
        samples_per_cluster = max(1, int(sample_size / max(1, len(unique_clusters))))
        sampled_data = []

        if filter_label and trainer.get_clf():
            df["predicted_label"] = trainer.get_inference(df)

        for cluster in unique_clusters:
            cluster_data = df[df["label_cluster"] == cluster]
            if cluster_data.empty:
                continue

            if filter_label and "predicted_label" in cluster_data.columns:
                pos = cluster_data[cluster_data["predicted_label"] == 1]
                neg = cluster_data[cluster_data["predicted_label"] == 0]
                n_pos = samples_per_cluster // 2
                pos_sample = get_sample(pos, n_pos)
                neg_sample = get_sample(neg, samples_per_cluster - len(pos_sample))
                chunk = pd.concat([pos_sample, neg_sample], ignore_index=True)
                if chunk.empty:
                    chunk = get_sample(cluster_data, samples_per_cluster)
            else:
                chunk = get_sample(cluster_data, samples_per_cluster)

            if not chunk.empty:
                sampled_data.append(chunk)

        if not sampled_data:
            sampled = get_sample(df, sample_size)
        else:
            sampled = pd.concat(sampled_data, ignore_index=True)
            if len(sampled) > sample_size:
                sampled = sampled.sample(sample_size, random_state=42).reset_index(drop=True)

        self.selected_ids.update(sampled["id"].astype(str).tolist())
        with open("selected_ids.txt", "w") as f:
            f.write("\\n".join(map(str, self.selected_ids)))

        return sampled, "random"
'''
with open(os.path.join(BASE_PATH, "random_sampling.py"), "w", encoding="utf-8") as f:
    f.write(random_code)

print("Wrote random_sampling.py")

Wrote random_sampling.py


In [19]:
thompson_code = r'''
from typing import Any
import numpy as np
from scipy.stats import beta
import pandas as pd


class ThompsonSampler:
    def __init__(self, n_bandits, alpha=0.5, beta=0.5, decay=0.99):
        self.n_bandits = n_bandits
        self.wins = np.zeros(n_bandits)
        self.losses = np.zeros(n_bandits)
        self.alpha = alpha
        self.beta = beta
        self.decay = decay

        try:
            loaded_ids = np.loadtxt("selected_ids.txt", dtype=str)
            if np.isscalar(loaded_ids):
                loaded_ids = np.array([loaded_ids])
            self.selected_ids = set(loaded_ids.tolist())
        except Exception:
            self.selected_ids = set()

        try:
            self.wins = np.loadtxt("wins.txt")
            self.losses = np.loadtxt("losses.txt")
            if np.isscalar(self.wins):
                self.wins = np.array([self.wins])
            if np.isscalar(self.losses):
                self.losses = np.array([self.losses])
        except Exception:
            self.wins = np.zeros(n_bandits)
            self.losses = np.zeros(n_bandits)

    def choose_bandit(self, exclude_bandits=None):
        if exclude_bandits is None:
            exclude_bandits = set()

        betas = beta(self.wins + self.alpha, self.losses + self.beta)
        sampled_rewards = betas.rvs(size=self.n_bandits)

        for b in exclude_bandits:
            if 0 <= b < self.n_bandits:
                sampled_rewards[b] = -1

        return int(np.argmax(sampled_rewards))

    def update(self, chosen_bandit, reward_difference):
        if isinstance(chosen_bandit, str):
            return

        if reward_difference > 0:
            self.wins[chosen_bandit] += 1
        else:
            self.losses[chosen_bandit] += 1

        self.wins *= self.decay
        self.losses *= self.decay

        np.savetxt("wins.txt", self.wins)
        np.savetxt("losses.txt", self.losses)

    def get_sample_data(self, df, sample_size, filter_label: bool, trainer: Any):
        def sample_from_df(df_in, n):
            if df_in.empty:
                return pd.DataFrame()
            return df_in.sample(min(n, len(df_in)), random_state=42)

        df = df[~df["id"].isin(self.selected_ids)].copy()
        if df.empty:
            raise ValueError("No unlabeled data remaining to sample from.")

        data = pd.DataFrame()
        chosen_bandit = None

        # --- Filtered mode: try a few distinct clusters first ---
        if filter_label and trainer.get_clf():
            tried_bandits = set()
            max_filtered_attempts = min(3, self.n_bandits)

            for _ in range(max_filtered_attempts):
                chosen_bandit = self.choose_bandit(exclude_bandits=tried_bandits)
                tried_bandits.add(chosen_bandit)

                print(f"Chosen bandit {chosen_bandit}")
                bandit_df = df[df["label_cluster"] == chosen_bandit].copy()
                print(f"length of bandit {len(bandit_df)}")

                if bandit_df.empty:
                    continue

                bandit_df["predicted_label"] = trainer.get_inference(bandit_df)
                print("inference results")
                print(bandit_df["predicted_label"].value_counts())

                pos = bandit_df[bandit_df["predicted_label"] == 1]
                neg = bandit_df[bandit_df["predicted_label"] == 0]

                if not pos.empty:
                    n_pos = sample_size // 2
                    pos_data = sample_from_df(pos, n_pos)
                    neg_data = sample_from_df(neg, sample_size - len(pos_data))
                    data = pd.concat([pos_data, neg_data]).sample(frac=1, random_state=42)
                    break
                else:
                    print("no predicted positives in chosen bandit, trying a different bandit")

            # Fallback: if filtered mode failed, sample unfiltered from the best remaining bandit
            if data.empty:
                print("No predicted positives found after trying multiple bandits. Falling back to unfiltered Thompson sampling.")
                fallback_bandit = self.choose_bandit()
                chosen_bandit = fallback_bandit
                bandit_df = df[df["label_cluster"] == chosen_bandit].copy()
                print(f"Fallback bandit {chosen_bandit}")
                print(f"length of fallback bandit {len(bandit_df)}")

                if not bandit_df.empty:
                    data = sample_from_df(bandit_df, sample_size)

        # --- Standard unfiltered mode ---
        else:
            chosen_bandit = self.choose_bandit()
            print(f"Chosen bandit {chosen_bandit}")

            bandit_df = df[df["label_cluster"] == chosen_bandit].copy()
            print(f"length of bandit {len(bandit_df)}")

            if not bandit_df.empty:
                data = sample_from_df(bandit_df, sample_size)

        # --- Final fallback if chosen cluster is empty or sampling still failed ---
        if data.empty:
            print("Falling back to random sampling from remaining data.")
            data = df.sample(min(sample_size, len(df)), random_state=42)
            chosen_bandit = "fallback_random"

        self.selected_ids.update(data["id"].astype(str).tolist())
        with open("selected_ids.txt", "w") as f:
            f.write("\\n".join(sorted(self.selected_ids)))

        return data, chosen_bandit
'''
with open(os.path.join(BASE_PATH, "thompson_sampling.py"), "w", encoding="utf-8") as f:
    f.write(thompson_code)

print("Wrote thompson_sampling.py")

Wrote thompson_sampling.py


In [20]:
fine_tune_code = r'''
from typing import Any, Optional, Dict

from transformers import Trainer, TrainingArguments, BertTokenizer, DataCollatorWithPadding, BertForSequenceClassification, TrainerCallback
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datasets import Dataset, DatasetDict
import torch
import pandas as pd
from torch import nn


class BertFineTuner:
    def __init__(self, model_name: Optional[str], training_data: Optional[pd.DataFrame], test_data: Optional[pd.DataFrame], learning_rate=2e-5, dropout=0.2):
        self.base_model = model_name
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.last_model_acc: Dict[str, float] = None
        self.training_data = training_data
        self.test_data = test_data
        self.trainer = None
        self.run_clf = False
        self.learning_rate = learning_rate
        self.weight_decay = 0.01

        model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
        if dropout:
            model.config.hidden_dropout_prob = dropout
            model.config.attention_probs_dropout_prob = dropout

        self.model = model
        self.model.to(self.device)

    def set_clf(self, set_value: bool):
        self.run_clf = set_value

    def get_clf(self):
        return self.run_clf

    def get_last_model_acc(self):
        return self.last_model_acc

    def get_base_model(self):
        return self.base_model

    def _text_col(self, df):
        if "training_text" in df.columns:
            return "training_text"
        return "title"

    def create_dataset(self, train, test):
        text_col_train = self._text_col(train)
        text_col_test = self._text_col(test)

        dataset_train = Dataset.from_pandas(
            train[[text_col_train, "label"]].rename(columns={text_col_train: "text"})
        )
        dataset_val = Dataset.from_pandas(
            test[[text_col_test, "label"]].rename(columns={text_col_test: "text"})
        )

        dataset = DatasetDict()
        dataset["train"] = dataset_train
        dataset["val"] = dataset_val

        def tokenize_function(element):
            return self.tokenizer(
                element["text"],
                padding="max_length",
                truncation=True,
                max_length=512
            )

        tokenized_data = dataset.map(tokenize_function, batched=True)
        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        return tokenized_data, data_collator

    def create_test_dataset(self, df: pd.DataFrame) -> Dataset:
        text_col = self._text_col(df)
        test_dataset = Dataset.from_pandas(
            df[[text_col]].rename(columns={text_col: "text"})
        )

        dataset = DatasetDict()
        dataset["test"] = test_dataset

        def tokenize_function(element):
            return self.tokenizer(
                element["text"],
                padding="max_length",
                truncation=True,
                max_length=512
            )

        tokenized_data = dataset.map(tokenize_function, batched=True)
        return tokenized_data

    @staticmethod
    def compute_metrics(pred):
        labels = pred.label_ids
        preds = pred.predictions.argmax(-1)

        return {
            "accuracy": accuracy_score(labels, preds),

            "precision": precision_score(labels, preds, average="weighted", zero_division=0),
            "recall": recall_score(labels, preds, average="weighted", zero_division=0),
            "f1": f1_score(labels, preds, average="weighted", zero_division=0),

            "precision_weighted": precision_score(labels, preds, average="weighted", zero_division=0),
            "recall_weighted": recall_score(labels, preds, average="weighted", zero_division=0),
            "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),

            "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
            "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),

            "precision_pos": precision_score(labels, preds, average="binary", pos_label=1, zero_division=0),
            "recall_pos": recall_score(labels, preds, average="binary", pos_label=1, zero_division=0),
            "f1_pos": f1_score(labels, preds, average="binary", pos_label=1, zero_division=0),
        }

    def train_data(self, df, still_unbalenced):
        early_stopping_callback = EarlyStoppingCallback(patience=5, log_dir="./log")
        tokenized_data, data_collator = self.create_dataset(df, self.test_data)

        training_args = TrainingArguments(
            output_dir="results",
            eval_strategy="epoch",
            save_strategy="epoch",
            metric_for_best_model="eval_accuracy",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=5,
            learning_rate=self.learning_rate,
            weight_decay=self.weight_decay,
            save_total_limit=2,
            logging_steps=10,
            push_to_hub=False,
            load_best_model_at_end=False,
            report_to=[]
        )

        trainer_class = MyTrainer if still_unbalenced else Trainer
        trainer = trainer_class(
            model=self.model,
            args=training_args,
            data_collator=data_collator,
            compute_metrics=BertFineTuner.compute_metrics,
            train_dataset=tokenized_data["train"],
            eval_dataset=tokenized_data["val"],
            callbacks=[early_stopping_callback]
        )

        trainer.train()
        print("Best checkpoint:", trainer.state.best_model_checkpoint)

        results = trainer.evaluate()
        print(results)

        self.trainer = trainer
        self.model = trainer.model
        return results, self.trainer

    def get_inference(self, df: pd.DataFrame) -> torch.Tensor:
        predicted_labels = []
        chunk_size = 10000
        total_records = len(df)
        start_index = 0

        while start_index < total_records:
            end_index = min(start_index + chunk_size, total_records)
            chunk = df[start_index:end_index]
            test_dataset = self.create_test_dataset(chunk)
            predictions = self.trainer.predict(test_dataset["test"])
            prediction_scores = predictions.predictions
            batch_predicted_labels = torch.argmax(torch.tensor(prediction_scores), dim=1)
            predicted_labels.append(batch_predicted_labels)
            start_index = end_index

        return torch.cat(predicted_labels)

    def save_model(self, path: str):
        if self.trainer is not None:
            self.trainer.save_model(path)

    def update_model(self, model_name, model_acc, save_model: bool):
        if save_model and self.trainer is not None:
            self.save_model(model_name)

        self.last_model_acc = {model_name: model_acc}
        self.base_model = model_name
        # Do not reload checkpoint here.


class MyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=torch.tensor([0.2, 0.8], device=model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=5, log_dir=None):
        self.patience = patience
        self.best_loss = float("inf")
        self.wait = 0
        self.log_dir = log_dir

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if state.is_world_process_zero and state.log_history:
            current_loss = None
            for log_entry in reversed(state.log_history):
                if "eval_loss" in log_entry:
                    current_loss = log_entry["eval_loss"]
                    break

            if current_loss is not None:
                if current_loss < self.best_loss:
                    self.best_loss = current_loss
                    self.wait = 0
                else:
                    self.wait += 1
                    if self.wait >= self.patience:
                        control.should_training_stop = True

                if self.log_dir:
                    with open(f"{self.log_dir}/epoch_{state.epoch}.txt", "w") as f:
                        for log in state.log_history:
                            f.write(f"{log}\n")
'''
with open(os.path.join(BASE_PATH, "fine_tune.py"), "w", encoding="utf-8") as f:
    f.write(fine_tune_code)

print("Wrote fine_tune.py")

Wrote fine_tune.py


In [21]:

main_cluster_code = r'''
import argparse
import pandas as pd
import numpy as np
from labeling import Labeling
from random_sampling import RandomSampler
from preprocessing import TextPreprocessor
from fine_tune import BertFineTuner
from thompson_sampling import ThompsonSampler
import nltk
import json
nltk.download("punkt")

import os
from LDA import LDATopicModel

def str2bool(v):
    if isinstance(v, bool):
        return v
    if v is None:
        return False
    v = str(v).strip().lower()
    if v in {"true", "1", "yes", "y"}:
        return True
    if v in {"false", "0", "no", "n"}:
        return False
    raise argparse.ArgumentTypeError(f"Boolean value expected, got: {v}")

def main():
    parser = argparse.ArgumentParser(prog="Sampling fine-tuning", description="Perform Sampling and fine tune")
    parser.add_argument("-sampling", type=str, required=False, help="Name of sampling method")
    parser.add_argument("-sample_size", type=int, required=False, help="sample size")
    parser.add_argument("-filter_label", type=str2bool, required=False, help="use model clf results to filter data")
    parser.add_argument("-balance", type=str2bool, required=False, help="balance positive and neg sample")
    parser.add_argument("-model_finetune", type=str, required=False, help="model base for fine tune")
    parser.add_argument("-labeling", type=str, required=False, help="Model to be used for labeling or file if label already on file")
    parser.add_argument("-baseline", type=float, required=False, help="The initial baseline metric")
    parser.add_argument("-filename", type=str, required=False, help="The initial file to be used")
    parser.add_argument("-model", type=str, required=False, help="The type of model to be finetune")
    parser.add_argument("-metric", type=str, required=False, help="The type of metric to be used for baseline")
    parser.add_argument("-val_path", type=str, required=False, help="path to validation")
    parser.add_argument("-cluster_size", type=int, required=False, help="number of clusters")
    parser.add_argument("-target_class", type=str, required=False, default="rec.autos")
    parser.add_argument("-few_shot_path", type=str, required=False, default=None)
    parser.add_argument("-hf_model_id", type=str, required=False, default="Qwen/Qwen2.5-3B-Instruct")
    parser.add_argument("-max_iterations", type=int, required=False, default=2)

    args = parser.parse_args()

    sampling = args.sampling
    sample_size = args.sample_size
    filter_label = args.filter_label
    balance = args.balance
    model_finetune = args.model_finetune
    labeling = args.labeling
    baseline = args.baseline
    filename = args.filename
    model = args.model
    metric = args.metric
    validation_path = args.val_path
    cluster_size = args.cluster_size
    target_class = args.target_class
    few_shot_path = args.few_shot_path
    hf_model_id = args.hf_model_id
    max_iterations = args.max_iterations

    preprocessor = TextPreprocessor()

    validation = pd.read_csv(validation_path)
    validation = preprocessor.preprocess_df(validation)
    validation["training_text"] = validation["clean_title"] if "clean_title" in validation.columns else validation["title"]

    os.makedirs("models", exist_ok=True)
    os.makedirs("data", exist_ok=True)
    os.makedirs("log", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    try:
        data = pd.read_csv(filename + "_lda.csv")
        n_cluster = data["label_cluster"].value_counts().count()
        print("using data saved on disk")
    except Exception:
        print("Creating LDA")
        data = pd.read_csv(filename + ".csv")
        data = preprocessor.preprocess_df(data)
        lda_topic_model = LDATopicModel(num_topics=cluster_size)
        topics = lda_topic_model.fit_transform(data["clean_title"].to_list())
        data["label_cluster"] = topics
        n_cluster = data["label_cluster"].value_counts().count()
        print(n_cluster)
        data.to_csv(filename + "_lda.csv", index=False)
        print("LDA created")

    if model == "text":
        trainer = BertFineTuner(model_finetune, None, validation)
    else:
        raise ValueError("Currently only text model is supported")

    if sampling == "thompson":
        sampler = ThompsonSampler(n_cluster)
    elif sampling == "random":
        sampler = RandomSampler(n_cluster)
    else:
        raise ValueError("Choose one of thompson or random")

    for i in range(max_iterations):
        sample_data, chosen_bandit = sampler.get_sample_data(data, sample_size, filter_label, trainer)

        if labeling != "file":
            labeler = Labeling(
                label_model=labeling,
                target_class=target_class,
                model_id=hf_model_id,
                few_shot_path=few_shot_path
            )
            labeler.set_model()

            df = labeler.generate_inference_data(sample_data, "clean_title")
            print("df for inference created")
            df["answer"] = df.apply(lambda x: labeler.predict_animal_product(x), axis=1)
            df["answer"] = df["answer"].astype(str).str.strip()
            df["pseudo_label"] = np.where(df["answer"] == "1", 1, 0)
            df["label"] = df["pseudo_label"]
            df["training_text"] = df["title"]

            if os.path.exists(f"{filename}_data_labeled.csv"):
                train_data = pd.read_csv(f"{filename}_data_labeled.csv")
                train_data = pd.concat([train_data, df], ignore_index=True)
                train_data.to_csv(f"{filename}_data_labeled.csv", index=False)
            else:
                df.to_csv(f"{filename}_data_labeled.csv", index=False)

            try:
                del labeler.model
                del labeler.tokenizer
            except Exception:
                pass
            del labeler

            try:
                import gc
                import torch
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass
        else:
            df = sample_data
            if "training_text" not in df.columns:
                df["training_text"] = df["clean_title"] if "clean_title" in df.columns else df["title"]

        if "true_label" in df.columns and "pseudo_label" in df.columns:
            agreement = (df["true_label"].astype(float).fillna(-1).astype(int) == df["pseudo_label"].astype(int)).mean()
            print(f"Pseudo-label agreement with true_label on sampled batch: {agreement:.3f}")

        print(df["label"].value_counts())

        if os.path.exists("positive_data.csv"):
            pos = pd.read_csv("positive_data.csv")
            df = pd.concat([df, pos], ignore_index=True).sample(frac=1, random_state=42)
            print(f"adding positive data: {df['label'].value_counts()}")

        if balance:
            if len(df[df["label"] == 1]) > 0:
                unbalanced = len(df[df["label"] == 0]) / len(df[df["label"] == 1]) > 2
                if unbalanced:
                    label_counts = df["label"].value_counts()
                    min_count = min(label_counts)
                    balanced_df = pd.concat([
                        df[df["label"] == 0].sample(min_count * 2, random_state=42),
                        df[df["label"] == 1].sample(min_count, random_state=42)
                    ])
                    df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
                    print(f"Balanced data: {df.label.value_counts()}")

        model_name = trainer.get_base_model()
        model_results = trainer.get_last_model_acc()
        if model_results:
            baseline = model_results[model_name]
            print(f"previous model {metric} metric baseline of: {baseline}")
        else:
            print(f"Starting with metric {metric} baseline {baseline}")
        print("Starting training")

        try:
            still_unbalenced = len(df[df["label"] == 0]) / len(df[df["label"] == 1]) >= 2
        except Exception:
            still_unbalenced = True
        print(f"Unbalanced? {still_unbalenced}")

        results, huggingface_trainer = trainer.train_data(df, still_unbalenced)
        reward_difference = results[f"eval_{metric}"] - baseline

        if reward_difference > 0:
            print(f"Model improved with {reward_difference}")
            model_name = f"models/fine_tunned_{i}_bandit_{chosen_bandit}"
            trainer.update_model(model_name, results[f"eval_{metric}"], save_model=True)
            if os.path.exists(f"{filename}_training_data.csv"):
                train_data = pd.read_csv(f"{filename}_training_data.csv")
                df = pd.concat([train_data, df], ignore_index=True)
            df.to_csv(f"{filename}_training_data.csv", index=False)
            if os.path.exists("positive_data.csv"):
                os.remove("positive_data.csv")
            if filter_label:
                trainer.set_clf(True)
        else:
            trainer.update_model(model_name, baseline, save_model=False)
            if os.path.exists("positive_data.csv"):
                positive = pd.read_csv("positive_data.csv")
                df = df[df["label"] == 1]
                df = pd.concat([df, positive], ignore_index=True).drop_duplicates()
            df[df["label"] == 1].to_csv("positive_data.csv", index=False)

        if os.path.exists(f"{filename}_model_results.json"):
            with open(f"{filename}_model_results.json", "r") as file:
                existing_results = json.load(file)
        else:
            existing_results = {}

        if existing_results.get(str(chosen_bandit)):
            existing_results[str(chosen_bandit)].append(results)
        else:
            existing_results[str(chosen_bandit)] = [results]

        with open(f"{filename}_model_results.json", "w") as file:
            json.dump(existing_results, file, indent=4)

        if sampling == "thompson":
            sampler.update(chosen_bandit if chosen_bandit != "fallback_random" else 0, reward_difference)

    if hasattr(sampler, "wins") and hasattr(sampler, "losses"):
        print("Bandit with highest expected improvement:", np.argmax(sampler.wins / np.maximum(sampler.wins + sampler.losses, 1e-8)))
        print(sampler.wins)
        print(sampler.losses)

if __name__ == "__main__":
    main()
'''
with open(os.path.join(BASE_PATH, "main_cluster.py"), "w", encoding="utf-8") as f:
    f.write(main_cluster_code)

print("Wrote main_cluster.py")


Wrote main_cluster.py


## 6) Reset state files before a fresh run

This step is important.

The pipeline stores:
- `selected_ids.txt`
- `wins.txt`
- `losses.txt`
- `positive_data.csv`

If you rerun without clearing them, sampling may behave strangely or appear to repeat old state.

In [95]:
for fname in [
    "selected_ids.txt",
    "wins.txt",
    "losses.txt",
    "positive_data.csv",
    f"train_inner_20news_{TARGET_SLUG}_model_results.json",
    f"train_inner_20news_{TARGET_SLUG}_training_data.csv",
    f"train_inner_20news_{TARGET_SLUG}_data_labeled.csv",
    f"train_inner_20news_{TARGET_SLUG}_lda.csv"
]:
    path = os.path.join(BASE_PATH, fname)
    if os.path.exists(path):
        os.remove(path)
        print("Removed:", path)

## 7) Sanity-check the Python files

This catches syntax problems before the long run starts.

In [100]:
import subprocess

result = subprocess.run(
    ["python", "-m", "py_compile",
     "preprocessing.py", "LDA.py", "labeling.py",
     "random_sampling.py", "thompson_sampling.py",
     "fine_tune.py", "main_cluster.py"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("All files compiled successfully.")
else:
    print("Compilation failed:")
    print(result.stderr)


All files compiled successfully.


## 8) Run the pipeline on the small debug dataset first

This is now set up for `rec.autos`.

Before running the full pipeline, update the curated few-shot IDs and, if needed, refine the `rec.autos` prompt in `labeling.py`.

In [104]:
import importlib
import pandas as pd
import labeling
importlib.reload(labeling)
from labeling import Labeling

# Clear rec.autos positives
hard_pos_ids = [
    103723,  # Viper car alarms
    103103,  # insurance says car is totalled
    103760,  # advice on used car
    103193,  # dealer cheated me / odometer
    103175,  # insurance companies
    102759,  # changing oil by self
    103039,  # aftermarket exhausts for BMW 320i
    103112,  # differences in insurance companies' rates
    101608,  # storing a car long term
    103739,  # Ford Probe opinions
]

# Hard negatives
hard_neg_ids = [
    104887,  # rec.motorcycles dogs
    105142,  # rec.motorcycles flame
    38572,   # comp.graphics paintbrush/scanner
    9522,    # windows graphics card
    15331,   # sci.crypt random bits
    83490,   # talk.religion.misc
    74737,   # misc.forsale discman
    76212,   # misc.forsale modem
]

bench_ids = hard_pos_ids + hard_neg_ids

full_df = pd.read_csv("train_inner_20news_rec_autos.csv")
bench_df = full_df[full_df["id"].astype(int).isin(bench_ids)].copy()
bench_df["clean_title"] = bench_df["title"]

labeler = Labeling(
    label_model="qwen",
    target_class="rec.autos",
    model_id="Qwen/Qwen2.5-3B-Instruct",
    few_shot_path="few_shot_examples_rec_autos.json"
)
labeler.set_model()

infer_df = labeler.generate_inference_data(bench_df, "title")
infer_df["pred_label"] = infer_df.apply(labeler.predict_animal_product, axis=1).astype(int)

result = infer_df[["id", "title", "true_label", "pred_label"]].copy()
result = result.rename(columns={"true_label": "label"})

false_pos = result[(result["label"] == 0) & (result["pred_label"] == 1)].copy()
false_neg = result[(result["label"] == 1) & (result["pred_label"] == 0)].copy()

tp = ((result["label"] == 1) & (result["pred_label"] == 1)).sum()
fp = ((result["label"] == 0) & (result["pred_label"] == 1)).sum()
fn = ((result["label"] == 1) & (result["pred_label"] == 0)).sum()
tn = ((result["label"] == 0) & (result["pred_label"] == 0)).sum()

pos_recall = tp / max((result["label"] == 1).sum(), 1)
neg_acc = tn / max((result["label"] == 0).sum(), 1)
pos_precision = tp / max(tp + fp, 1)
pos_f1 = 0.0 if (pos_precision + pos_recall) == 0 else 2 * pos_precision * pos_recall / (pos_precision + pos_recall)

print("\nFalse Positives:")
if false_pos.empty:
    print("None")
else:
    print(false_pos[["id", "title"]].to_string(index=False))

print("\nFalse Negatives:")
if false_neg.empty:
    print("None")
else:
    print(false_neg[["id", "title"]].to_string(index=False))

print("\nSummary:")
summary_df = pd.DataFrame([{
    "tp": tp,
    "fp": fp,
    "fn": fn,
    "tn": tn,
    "positive_precision": pos_precision,
    "positive_recall": pos_recall,
    "positive_f1": pos_f1,
    "negative_accuracy": neg_acc,
    "predicted_ones": int((result["pred_label"] == 1).sum())
}])
print(summary_df)

print("\nFull results:")
print(result.sort_values(["label", "id"], ascending=[False, True]).to_string(index=False))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen model loaded

False Positives:
None

False Negatives:
    id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                title
103112 differences in insurance companies' rates I am considering buying an new car, so I called three insurance 

In [98]:
import importlib
import pandas as pd
import torch
import labeling
importlib.reload(labeling)
from labeling import Labeling

inspect_ids = [
    103039,  # aftermarket exhausts for BMW 320i
    101608,  # storing a car long term
    103112,  # insurance companies' rates
    103739,  # Ford Probe opinions
    103760,  # advice on used car
    103103,  # insurance says car is totalled
    103723,  # Viper car alarms
    38572,   # comp.graphics negative
    104887,  # rec.motorcycles negative
]

full_df = pd.read_csv("train_inner_20news_rec_autos.csv")
inspect_df = full_df[full_df["id"].astype(int).isin(inspect_ids)].copy()
inspect_df["clean_title"] = inspect_df["title"]

labeler = Labeling(
    label_model="qwen",
    target_class="rec.autos",
    model_id="Qwen/Qwen2.5-3B-Instruct",
    few_shot_path="few_shot_examples_rec_autos.json"
)
labeler.set_model()

infer_df = labeler.generate_inference_data(inspect_df, "title")

for _, row in infer_df.iterrows():
    user_prompt = row["text"]

    messages = [
        {
            "role": "system",
            "content": "You are a binary classifier. Output exactly one character: 1 or 0. Do not output any words."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    text = labeler.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = labeler.tokenizer(
        [text],
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(labeler.model.device)

    input_len = model_inputs.input_ids.shape[1]

    with torch.inference_mode():
        outputs = labeler.model.generate(
            **model_inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=labeler.tokenizer.eos_token_id
        )

    generated_ids = outputs[:, input_len:]
    raw_response = labeler.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed_label = labeler._extract_label(raw_response)

    print("=" * 100)
    print("ID:", row["id"])
    print("TRUE LABEL:", row["true_label"])
    print("TITLE:", row["title"])
    print("RAW RESPONSE:", repr(raw_response))
    print("PARSED LABEL:", parsed_label)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen model loaded
ID: 38572
TRUE LABEL: 0
TITLE: PC PAINTBRUSH IV+ I am trying to configure Zsoft's PC Paintbrush IV+ for use with my Logitech Scanman 32 (hand scanner), but I can't get Paintbrush to acknowledge the scanner. Is there anybody out there using Paintbrush with a scanner, if so, can you help me out? Thanks Luis Nobrega | The File Bank BBS - 619-728-4318 - PCBoard v.14.5a/E10 - USR HST & DS | | 8 nodes / RIME / Internet / Largest Clipper file collection in the world |
RAW RESPONSE: '0'
PARSED LABEL: 0
ID: 104887
TRUE LABEL: 0
TITLE: Re: dogs + +>This tactic depends for its effectiveness on the dog's conformance to +>a "psychological norm" that may not actually apply to a particular dog. +>I've tried it with some success before, but it won't work on a Charlie Manson +>dog or one that's really, *really* stupid. A large Irish Setter taught me +>this in *my* yard (apparently HIS territory) one day. I'm sure he was playing +>a game with me. The game was probably "Kill the VERY AN

In [24]:
%cd "/content/drive/MyDrive/Colab Notebooks/Data_Engineering_project"

/content


This version keeps only the columns `id`, `title`, `subject`, `label`, and `newsgroup` in the generated train/val/test CSVs. The pipeline uses the shortened cleaned `title` text for clustering, Qwen labeling, and debug BERT runs. It is now configured for `rec.autos`.

In [105]:
!python main_cluster.py \
  -sample_size 150 \
  -filename "train_inner_20news_rec_autos" \
  -val_path "val_20news_rec_autos.csv" \
  -balance False \
  -sampling "thompson" \
  -filter_label True \
  -model_finetune "bert-base-uncased" \
  -labeling "qwen" \
  -model "text" \
  -baseline 0.5 \
  -metric "f1_pos" \
  -cluster_size 10 \
  -target_class "rec.autos" \
  -few_shot_path "few_shot_examples_rec_autos.json" \
  -hf_model_id "Qwen/Qwen2.5-3B-Instruct" \
  -max_iterations 4

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Creating LDA
10
LDA created
Loading weights: 100% 199/199 [00:00<00:00, 21324.61it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

In [107]:
import pandas as pd

df = pd.read_csv("train_inner_20news_rec_autos_data_labeled.csv")
print(df["label"].value_counts(dropna=False))

if "true_label" in df.columns:
    print(df["true_label"].value_counts(dropna=False))
    print("Agreement:", (df["label"] == df["true_label"]).mean())

print(df[["title", "label"]].head(10))

label
0    507
1     93
Name: count, dtype: int64
true_label
0    494
1    106
Name: count, dtype: int64
Agreement: 0.885
                                               title  label
0  Re: BMW heated grips Someone once sold heated ...      0
1  Re: <<Pompous ass In article <1ql71pINN5ef@gap...      1
2  Re: Help with backpack Sanjay Sinha (sanjay@ki...      0
3  Re: Insurance discount My agent is Daniel Sui,...      0
4  Re: Countersteering_FAQ please post mjs>Well, ...      0
5  Re: GGRRRrrr!! Cages double-parking motorcycle...      0
6  Re: WACO burning Generally the ship sinks (sor...      0
7  Re: "Jump Starting" a Mac II Yes, 4 points, in...      0
8  Re: Donating organs I'm sure the Pittsburgh gr...      0
9  Re: GGRRRrrr!! Cages double-parking motorcycle...      0


In [90]:
import pandas as pd

df = pd.read_csv("train_inner_20news_rec_autos_data_labeled.csv")

false_negatives = df[(df["true_label"] == 1) & (df["label"] == 0)].copy()

print("Number of false negatives:", len(false_negatives))
print(false_negatives[["title"]].head(30).to_string(index=False))

Number of false negatives: 54
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [91]:
cols = [c for c in ["id", "title", "true_label", "label", "answer"] if c in false_negatives.columns]
print(false_negatives[cols].head(50).to_string(index=False))

    id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [92]:
for i, row in false_negatives.head(30).iterrows():
    print(f"\n--- False negative {i} ---")
    print(row["title"])


--- False negative 2 ---
Re: Dumbest automotive concepts of all time seningen@maserati.ross.com (Mike Seningen) The funny thing about the digital dash (87 T-bird) with the 85mph speedo limit was that if you pressed the button to convert to kilometers it would read all the way up to 187kph. At this point the stock anemic 302 would get short of breath. This of course was equivalent to about 116mph (hehe).I bet I really coulda confused this thing if I'd toyed with the engine and rolled the stupid thing (the digits were limited to 199). I've gotta agree with ya on the analog clock w/digital dash though. My girlfriend had a '85 TurboCoupe with a digital clock and analog gauges/radio. Go figure... usenet@constellation.ecn.uoknor.edu (Usenet Administrator) I had a great feature on my T-bird.... I could pull the key out and leave the ignition on. This scared the hell out of me the first time it happened but I kinda grew to like it. Musta been a bad key copy or something. Mark Novakovic "There

## Optional next experiment

After the first small `rec.autos` run finishes and the pseudo-label agreement looks reasonable, try the same command with:

- `-filter_label True`

That can help once the first fine-tuned classifier becomes informative enough to identify likely positives inside the chosen cluster.

In [47]:
df = pd.read_csv("train_inner_20news_rec_autos.csv")
print(df["newsgroup"].value_counts().head(20))
print(df["label"].value_counts())
print(df[df["label"] == 1][["id", "newsgroup", "title"]].head(20).to_string(index=False))
print(df[df["label"] == 0][["id", "newsgroup", "title"]].head(20).to_string(index=False))

newsgroup
comp.sys.ibm.pc.hardware    649
sci.crypt                   643
rec.autos                   634
soc.religion.christian      630
comp.os.ms-windows.misc     624
comp.windows.x              623
comp.sys.mac.hardware       614
comp.graphics               606
sci.med                     603
misc.forsale                596
rec.motorcycles             545
talk.politics.misc          518
alt.atheism                 505
rec.sport.hockey            481
talk.religion.misc          419
talk.politics.guns          333
sci.space                   311
talk.politics.mideast       221
rec.sport.baseball          168
sci.electronics             135
Name: count, dtype: int64
label
0    9224
1     634
Name: count, dtype: int64
    id newsgroup                                                                                                                                                                                                                                                                

In [68]:
import os
import pandas as pd

for f in sorted(os.listdir(".")):
    if f.endswith(".csv"):
        try:
            tmp = pd.read_csv(f, nrows=5)
            if "label_cluster" in tmp.columns:
                print("FOUND:", f)
                print(tmp.columns.tolist())
                display(tmp.head())
        except Exception:
            pass

FOUND: train_inner_20news_rec_autos_lda.csv
['id', 'title', 'subject', 'label', 'newsgroup', 'clean_title', 'label_cluster']


,id,title,subject,label,newsgroup,clean_title,label_cluster
0,54288,Re: ESPN Thumbs Up your $%#@*!! In article <19...,Re: ESPN Thumbs Up your $%#@*!!,0,rec.sport.hockey,Re: ESPN Thumbs Up your $%#@*!! In article <19...,4
1,54658,Re: Re: Guns GONE. Good Riddance ! / iftccu:ta...,Re: Re: Guns GONE. Good Riddance !,0,talk.politics.guns,Re: Re: Guns GONE. Good Riddance ! / iftccu:ta...,0
2,38572,PC PAINTBRUSH IV+ I am trying to configure Zso...,PC PAINTBRUSH IV+,0,comp.graphics,PC PAINTBRUSH IV+ I am trying to configure Zso...,1
3,83490,"Re: ""lds"" Rick's reply #Rick Anderson replied ...","Re: ""lds"" Rick's reply",0,talk.religion.misc,"Re: ""lds"" Rick's reply #Rick Anderson replied ...",7
4,104887,Re: dogs + +>This tactic depends for its effec...,Re: dogs,0,rec.motorcycles,Re: dogs + +>This tactic depends for its effec...,0


In [84]:
clustered_df = pd.read_csv("train_inner_20news_rec_autos_lda.csv")
print(clustered_df.columns.tolist())
print(clustered_df[["id", "newsgroup", "label", "label_cluster", "title"]].head(10).to_string(index=False))

['id', 'title', 'subject', 'label', 'newsgroup', 'clean_title', 'label_cluster']
    id               newsgroup  label  label_cluster                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [85]:
cluster_summary = (
    clustered_df.groupby("label_cluster")
    .agg(
        cluster_size=("label", "size"),
        num_positive=("label", "sum")
    )
)
cluster_summary["positive_rate"] = cluster_summary["num_positive"] / cluster_summary["cluster_size"]
print(cluster_summary.sort_values("positive_rate", ascending=False))

               cluster_size  num_positive  positive_rate
label_cluster                                           
6                       754           287       0.380637
8                       614            75       0.122150
0                      1166           121       0.103774
9                      1452            91       0.062672
4                       732            26       0.035519
5                       592             8       0.013514
2                       709             6       0.008463
3                       599             5       0.008347
7                      1462            10       0.006840
1                      1778             5       0.002812


In [38]:
top_clusters = (
    cluster_summary.sort_values("positive_rate", ascending=False)
    .head(3)
    .index
    .tolist()
)

for c in top_clusters:
    print("\n" + "=" * 90)
    print("CLUSTER", c)
    print("positive rate:", cluster_summary.loc[c, "positive_rate"])
    print(
        clustered_df[(clustered_df["label_cluster"] == c) & (clustered_df["label"] == 1)]
        [["id", "title"]]
        .head(15)
        .to_string(index=False)
    )


CLUSTER 4
positive rate: 0.2618801652892562
    id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     